In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from peft import LoraConfig, get_peft_model, TaskType

from dataclasses import dataclass
from typing import Optional, Dict, Any

from sklearn.metrics import mean_absolute_error, cohen_kappa_score

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)

In [2]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"   # khuyên dùng bản này cho mốc 3B
TRAIN_CSV = "/content/ielts_train_df.csv"
VAL_CSV   = "/content/ielts_val_df.csv"
TEST_CSV  = "/content/ielts_test_df.csv"

MAX_LENGTH = 1024
SEED = 42

BATCH_SIZE = 1
GRAD_ACCUM = 16

LR = 1e-5
EPOCHS = 7
WEIGHT_DECAY = 0.01
OUTPUT_DIR = "./qwen25_3b_ielts_4criteria"

TARGET_COLS = ["TR", "CC", "LR", "GRA"]

set_seed(SEED)

In [3]:
# --- CẬP NHẬT CELL 3: LÀM SẠCH DỮ LIỆU TRIỆT ĐỂ ---

# 1. Load dữ liệu gốc
train_df = pd.read_csv(TRAIN_CSV, engine="python", on_bad_lines="skip")
val_df = pd.read_csv(VAL_CSV, engine="python", on_bad_lines="skip") if os.path.exists(VAL_CSV) else None
test_df = pd.read_csv(TEST_CSV, engine="python", on_bad_lines="skip") if os.path.exists(TEST_CSV) else None

# 2. Tách tập validation nếu chưa có
if val_df is None:
    from sklearn.model_selection import train_test_split
    train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=SEED)

needed_cols = ["prompt", "essay"] + TARGET_COLS

def robust_clean_df(df):
    if df is None: return None
    # Chỉ lấy các cột cần thiết
    df = df[needed_cols].copy()

    # Ép kiểu nhãn sang numeric và loại bỏ NaN ở nhãn
    for col in TARGET_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=TARGET_COLS).reset_index(drop=True)

    # QUAN TRỌNG: Ép kiểu essay sang string và loại bỏ khoảng trắng thừa
    df['essay'] = df['essay'].astype(str).str.strip()

    # Loại bỏ các bài viết quá ngắn (dưới 20 ký tự thường là dữ liệu lỗi)
    df = df[df['essay'].str.len() > 20].reset_index(drop=True)

    # Giới hạn điểm số trong dải hợp lệ [0.0, 9.0] để tránh outlier làm nổ gradient
    for col in TARGET_COLS:
        df[col] = df[col].clip(0.0, 9.0)

    return df

# Áp dụng hàm làm sạch cho các tập dữ liệu
train_df = robust_clean_df(train_df)
val_df = robust_clean_df(val_df)
test_df = robust_clean_df(test_df)

print(f"Train shape: {train_df.shape if train_df is not None else 0}")
print(f"Val shape: {val_df.shape if val_df is not None else 0}")
print(train_df[TARGET_COLS].head())

Train shape: (6618, 6)
Val shape: (827, 6)
    TR   CC   LR  GRA
0  6.0  6.0  6.0  6.0
1  6.5  6.5  6.5  6.5
2  7.0  7.0  7.0  7.0
3  5.0  5.0  5.0  5.0
4  4.5  4.5  4.5  4.5


In [4]:
def build_input_text(row):
    prompt = str(row["prompt"]).strip()
    essay = str(row["essay"]).strip()
    return f"[PROMPT]\n{prompt}\n\n[ESSAY]\n{essay}"

train_df["text"] = train_df.apply(build_input_text, axis=1)
val_df["text"] = val_df.apply(build_input_text, axis=1)

if test_df is not None:
    test_df["text"] = test_df.apply(build_input_text, axis=1)

def make_labels(df):
    df = df.copy()
    df["labels"] = (df[TARGET_COLS] / 9.0).values.tolist()
    return df

train_df = make_labels(train_df)
val_df = make_labels(val_df)
if test_df is not None:
    test_df = make_labels(test_df)

In [5]:
train_ds = Dataset.from_pandas(train_df[["text", "labels"]], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[["text", "labels"]], preserve_index=False)

dataset_dict = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
})

if test_df is not None:
    test_ds = Dataset.from_pandas(test_df[["text", "labels"]], preserve_index=False)
    dataset_dict["test"] = test_ds

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

tokenized_ds = dataset_dict.map(tokenize_fn, batched=True)
tokenized_ds = tokenized_ds.remove_columns(["text"])
tokenized_ds.set_format(type="torch")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/6618 [00:00<?, ? examples/s]

Map:   0%|          | 0/827 [00:00<?, ? examples/s]

Map:   0%|          | 0/828 [00:00<?, ? examples/s]

In [7]:
def build_model(model_name: str, num_labels: int = 4):
    config = AutoConfig.from_pretrained(model_name)
    config.num_labels = num_labels
    config.problem_type = "regression"
    config.pad_token_id = tokenizer.pad_token_id

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        config=config,
        torch_dtype=torch.bfloat16,
    )

    model.config.pad_token_id = tokenizer.pad_token_id
    model.config.use_cache = False
    model.gradient_checkpointing_enable()

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model

model = build_model(MODEL_NAME, num_labels=len(TARGET_COLS))

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-3B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 3,694,592 || all params: 3,089,641,472 || trainable%: 0.1196


In [8]:
from dataclasses import dataclass
from typing import Any, Dict, List
import torch

@dataclass
class RegressionCollator:
    tokenizer: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        labels = torch.stack([
            f["labels"] if isinstance(f["labels"], torch.Tensor)
            else torch.tensor(f["labels"], dtype=torch.float)
            for f in features
        ]).float()

        batch = self.tokenizer.pad(
            [{k: v for k, v in f.items() if k != "labels"} for f in features],
            padding=True,
            return_tensors="pt"
        )
        batch["labels"] = labels
        return batch

data_collator = RegressionCollator(tokenizer)

In [9]:
def round_to_half(x):
    return np.round(x * 2) / 2

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.asarray(preds, dtype=np.float32)
    labels = np.asarray(labels, dtype=np.float32)

    preds = preds * 9.0
    labels = labels * 9.0

    preds = np.clip(preds, 0.0, 9.0)
    labels = np.clip(labels, 0.0, 9.0)

    preds_half = round_to_half(preds)

    maes = []
    qwks = []

    for i in range(labels.shape[1]):
        maes.append(mean_absolute_error(labels[:, i], preds_half[:, i]))

        y_true = np.round(labels[:, i] * 2).astype(int)
        y_pred = np.round(preds_half[:, i] * 2).astype(int)

        qwks.append(cohen_kappa_score(y_true, y_pred, weights="quadratic"))

    within_05 = np.mean(np.abs(preds_half - labels) <= 0.5)

    return {
        "mean_mae": float(np.mean(maes)),
        "mean_qwk": float(np.mean(qwks)),
        "tr_qwk": float(qwks[0]),
        "cc_qwk": float(qwks[1]),
        "lr_qwk": float(qwks[2]),
        "gra_qwk": float(qwks[3]),
        "within_0.5_acc": float(within_05),
    }

In [10]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=20,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    learning_rate=LR,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_ratio=0.1,

    load_best_model_at_end=True,
    metric_for_best_model="mean_qwk",
    greater_is_better=True,

    bf16=True,
    fp16=False,

    gradient_checkpointing=True,
    remove_unused_columns=False,
    report_to="none",
    save_total_limit=2,
    max_grad_norm=1.0,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [11]:
from transformers import get_cosine_schedule_with_warmup

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-5,
    weight_decay=0.01
)

num_update_steps_per_epoch = max(
    1, len(tokenized_ds["train"]) // (BATCH_SIZE * GRAD_ACCUM)
)

num_training_steps = num_update_steps_per_epoch * EPOCHS

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(num_training_steps * 0.1),
    num_training_steps=num_training_steps
)

In [19]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    optimizers=(optimizer, scheduler),
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=10)],
)

In [13]:
from torch.utils.data import DataLoader

debug_loader = DataLoader(
    tokenized_ds["train"],
    batch_size=2,
    collate_fn=data_collator
)

batch = next(iter(debug_loader))
print(batch.keys())
print(batch["labels"])
print(batch["labels"].shape)
print(batch["labels"].dtype)

KeysView({'input_ids': tensor([[ 42347,   3361,   2828,    921,   8441,   5837,    525,  10164,    264,
           6765,   3311,    315,   3220,    389,  12613,    862,  27550,    311,
           1896,    949,    304,   1045,  15245,   9833,  42582,     13,  25028,
          17585,    429,    432,   1035,    387,   2664,    421,   1493,   5837,
            646,   8329,    279,   3220,    389,   2841,    311,   1896,    949,
            304,   9833,     13,   2014,   1128,  12818,    653,    498,   7503,
            476,  28295,   1939,     58,   9996,   3022,    921,   3862,    525,
           2696,    315,   5837,    304,    279,   1879,    879,    525,   1667,
           2244,   3311,    315,   3220,    389,  42649,    315,    862,   7263,
             13,    358,   7503,    429,    807,   1265,   1191,  10164,   3220,
            389,   2841,  10552,   7488,    304,    429,   1616,    807,    686,
           1191,    279,  50375,    315,   4124,  17628,    304,    862,   2272,
     

In [21]:
trainer.train()

KeyboardInterrupt: 

In [15]:
val_metrics = trainer.evaluate(tokenized_ds["validation"])
print("Validation metrics:", val_metrics)

if "test" in tokenized_ds:
    test_metrics = trainer.evaluate(tokenized_ds["test"], metric_key_prefix="test")
    print("Test metrics:", test_metrics)

Validation metrics: {'eval_loss': 0.014094904996454716, 'eval_mean_mae': 0.8097037374973297, 'eval_mean_qwk': 0.5564910306933928, 'eval_tr_qwk': 0.5739220086449769, 'eval_cc_qwk': 0.558914494111865, 'eval_lr_qwk': 0.5637346307696589, 'eval_gra_qwk': 0.5293929892470701, 'eval_within_0.5_acc': 0.5465538089480049, 'eval_runtime': 67.2129, 'eval_samples_per_second': 12.304, 'eval_steps_per_second': 12.304, 'epoch': 7.0}


early stopping required metric_for_best_model, but did not find eval_mean_qwk so early stopping is disabled


Test metrics: {'test_loss': 0.014195536263287067, 'test_mean_mae': 0.8215579688549042, 'test_mean_qwk': 0.5515026260176427, 'test_tr_qwk': 0.5806396768642645, 'test_cc_qwk': 0.568888656996195, 'test_lr_qwk': 0.5760282028061675, 'test_gra_qwk': 0.48045396740394397, 'test_within_0.5_acc': 0.5428743961352657, 'test_runtime': 67.529, 'test_samples_per_second': 12.261, 'test_steps_per_second': 12.261, 'epoch': 7.0}


In [22]:
trainer.save_model(f"{OUTPUT_DIR}/best_model")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/best_model")

('./qwen25_3b_ielts_4criteria/best_model/tokenizer_config.json',
 './qwen25_3b_ielts_4criteria/best_model/chat_template.jinja',
 './qwen25_3b_ielts_4criteria/best_model/tokenizer.json')

In [26]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

OUTPUT_DIR = "./qwen25_3b_ielts_4criteria"

shutil.make_archive("/content/model_output", 'zip', OUTPUT_DIR)

shutil.copy("/content/model_output.zip",
            "/content/drive/MyDrive/model_output.zip")

print("Model zipped and saved to Google Drive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Model zipped and saved to Google Drive!


In [1]:
import os
import gc
import torch

# Nếu muốn train tiếp thêm, tăng tổng số epoch lên
# Ví dụ trước đó đã train 3 epoch, giờ muốn train tới epoch 6:
trainer.args.num_train_epochs = 20

# Tự tìm checkpoint mới nhất trong OUTPUT_DIR
checkpoint_dirs = [
    os.path.join(OUTPUT_DIR, d)
    for d in os.listdir(OUTPUT_DIR)
    if d.startswith("checkpoint-") and os.path.isdir(os.path.join(OUTPUT_DIR, d))
]

if not checkpoint_dirs:
    raise ValueError(f"Không tìm thấy checkpoint nào trong {OUTPUT_DIR}")

latest_checkpoint = max(checkpoint_dirs, key=lambda x: int(x.split("-")[-1]))
print("Resuming from:", latest_checkpoint)

gc.collect()
torch.cuda.empty_cache()

train_result = trainer.train(resume_from_checkpoint=latest_checkpoint)

print("Resume train xong.")
print(train_result)

NameError: name 'trainer' is not defined